In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn scipy

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

In [6]:
# --- CONFIGURAÇÃO INICIAL (Assumindo que o DF já existe) ---
# Caso precise carregar do zero, descomente as linhas abaixo:

arquivos = ['vendas_linha_petshop_2019.csv', 'vendas_linha_petshop_2020.csv',
             'vendas_linha_petshop_2021.csv', 'vendas_linha_petshop_2022.csv']
vendas_totais = pd.concat([pd.read_csv(f, sep=';', decimal=',', encoding='latin1') for f in arquivos])

# Limpeza básica necessária para os cálculos
vendas_totais['quantidade'] = pd.to_numeric(vendas_totais['quantidade'], errors='coerce').fillna(1)
vendas_totais['valor'] = pd.to_numeric(vendas_totais['valor'], errors='coerce').fillna(0)

In [7]:
# RESPOTA QUESTÃO 2: Outliers em Quantidade

print("--- ANÁLISE DA QUESTÃO 2 ---")

# 1. Identificação de Outliers via Z-Score
z_scores_qtd = np.abs(stats.zscore(vendas_totais['quantidade']))
outliers_qtd = vendas_totais[z_scores_qtd > 3]

print(f"Total de outliers encontrados (z > 3): {len(outliers_qtd)}")
print(f"Quantidade máxima encontrada: {vendas_totais ['quantidade'].max()}")
print(f"Quantidade mediana: {vendas_totais ['quantidade'].median()}")

--- ANÁLISE DA QUESTÃO 2 ---
Total de outliers encontrados (z > 3): 4762
Quantidade máxima encontrada: 110.0
Quantidade mediana: 1.0


In [11]:
# 2. Identificação das vendas associadas
# Verificando se outliers de quantidade têm valores totais inconsistentes
print("\nExemplo de vendas com oitliers de quantidade:")
print(outliers_qtd[['produto', 'quantidade', 'valor_total_bruto']].head())


Exemplo de vendas com oitliers de quantidade:
                                               produto  quantidade  \
30                             Bola Pet Vinil Big Blue        71.0   
156                          Roupa para Cão Billaboard        70.0   
165                          Roupa para Cão Billaboard        70.0   
183                      Vitamina E Granulado BigForce        70.0   
228  Suplemento Alimentar Glutamina Mundo Animal Nu...        72.0   

     valor_total_bruto  
30              1491.0  
156             2240.0  
165             3010.0  
183             5040.0  
228             3096.0  


In [12]:
# 3. Estimativa de variabilidade ignorando outliers (Winsorização)
# Aplicando corte nos percentis 5% e 95% para maior estabilidade

lower_bound = vendas_totais['quantidade'].quantile(0.05)
upper_bound = vendas_totais['quantidade'].quantile(0.95)
quantidade_wins = vendas_totais['quantidade'].clip(lower_bound, upper_bound)

std_robusto = quantidade_wins.std()
print(f"\nDesvio Padrão Robusto (Dados Winsorizados): {std_robusto:.2f}")


Desvio Padrão Robusto (Dados Winsorizados): 0.94


In [14]:
# RESPOSTA DA QUESTÃO 3: Testes de Hipótese (Média de Preço)

print("\n" + "="*30 + "\n")
print("--- ANÁLISE DA QUESTÃO 3 ---")

media_populacao = vendas_totais['valor'].mean()
alpha = 0.05

def realizar_teste_t(df, coluna_grupo, valor_coluna, media_ref):
  grupo = df[df[coluna_grupo] == valor_coluna]['valor'].dropna()
  # Teste t de uma amostra: Compara a média do grupo com a média da população
  t_stat, p_val = stats.ttest_1samp(grupo, media_ref)
  return p_val

# A) Diferença por Região
print("A) Comparação por Região:")
regioes = vendas_totais['regiao_pais'].unique()
for r in regioes:
  p = realizar_teste_t(vendas_totais, 'regiao_pais', r, media_populacao)
  status = "SIGNIFICATIVA" if p < alpha else "não significativa"
  print(f"- Região {r:12} | p-value: {p:.4f} | Diferença {status}")

# B) Diferença por modalidade de pagamento
print("\nB) Comparação por Modalidade de Pagamento:")
pagamentos = vendas_totais['formapagto'].unique()
for pg in pagamentos:
  p = realizar_teste_t(vendas_totais, 'formapagto', pg, media_populacao)
  status = "SIGNIFICATIVA" if p < alpha else "não significativa"
  print(f"- pagamento {pg:12} | p-value: {p:.4f} | Diferença {status}")



--- ANÁLISE DA QUESTÃO 3 ---
A) Comparação por Região:
- Região Norte        | p-value: 0.9864 | Diferença não significativa
- Região Centro Oeste | p-value: 0.9049 | Diferença não significativa
- Região Nordeste     | p-value: 0.9939 | Diferença não significativa
- Região Sudeste      | p-value: 0.9907 | Diferença não significativa
- Região Sul          | p-value: 0.8731 | Diferença não significativa

B) Comparação por Modalidade de Pagamento:
- pagamento Dinheiro     | p-value: 0.7191 | Diferença não significativa
- pagamento Pix          | p-value: 0.8952 | Diferença não significativa
- pagamento Boleto Bancário | p-value: 0.6366 | Diferença não significativa
- pagamento Cartão Crédito | p-value: 0.1911 | Diferença não significativa
- pagamento Cartão Débito | p-value: 0.1706 | Diferença não significativa
